# Build a Resume AI Assistant with OpenAI, Gemini, and Pydantic  
In this Project we'll build a "Resume AI Assistant" which uses the power of LLMs like OpenAI's GPT, Google's Gemini, and Pydantic to help job seekers tailor their resume and generate custom cover letters for specific job applications.

## Learning Objectives  
- Build a powerful AI resume editor that can identify gaps between resume and the job description and tailor the resume and cover letter.
- Master Pydantic library for output validation.
- Learn how to generate parsed structured output from OpenAI with Pydantic.
- Learn how to develop a text change tracker using OpenAI.

## Pydantic Library  
- Pydantic is a Python library used to validate and parse data using Python type hints, it ensures that the data you get is clean, well-structured, and follows the rules you set.
- How it works:
    - You define a data structure using a class.
    - Pydantic ensures the data is correct (ypes, required fields, etc.)
    - It even converts data automatically if possible (e.g., turns strings into numbers or dates).

In [1]:
# Lets install and import Pydantic
# In Pydantic, BaseModel is the core class that you use to create data models.
# BaseModel is like a blueprint for structrued data. It defines the fields, their types, and automatically gives you data validation and type conversion capabilities.
%pip install pydantic
from pydantic import BaseModel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Let's create a new class named 'User'
# BaseModel is a special class from Pydantic that performs validation and parsing.
# Inside the class, we will declare name, age, and email along with their expected data types using Python type hints
# Pydantic's role is to validate that name is tr, age is an int, and so on.
# If you pass something wrong (like a string instead of a number), Pydantic raises an error.
class User(BaseModel):
    name: str
    age: int
    email: str

In [5]:
# Let's test it out with a valid (correct) input.
user = User(name = 'Mira', age = 30, email = 'mira@gmail.com')
print(user.json())

{"name":"Mira","age":30,"email":"mira@gmail.com"}


C:\Users\Paco_Minha\AppData\Local\Temp\ipykernel_2412\2270401647.py:3: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(user.json())


In [6]:
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

product = Product(name="Wireless Mouse", price = 29.99, in_stock=True)
print(product.json())

{"name":"Wireless Mouse","price":29.99,"in_stock":true}


C:\Users\Paco_Minha\AppData\Local\Temp\ipykernel_2412\826334947.py:7: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(product.json())


In [7]:
product = Product(name="Wireless Mouse", price = 10, in_stock=True)
print(product.json())

{"name":"Wireless Mouse","price":10.0,"in_stock":true}


C:\Users\Paco_Minha\AppData\Local\Temp\ipykernel_2412\1141446329.py:2: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(product.json())


## Generate Parsed Structured Output From OpenAI with Pydantic

In [10]:
# Install necessary libraries if running for the first time
%pip install openai google-generativeai python-dotenv ipython

# Import necessary libraries
import os
import google.generativeai as genai
from openai import OpenAI
from dotenv import load_dotenv
import json

# Import type hints that help describe kind of data your python functions or classes expect or return.
# List: A list of elements, all usually of the same data type.
# Example: List[int] means a list of integers like [1,2,3]

# Dict: A dictionary (key-value pairs).
# Example: Dict[str, int] means keys are strings and values are integer like {'a': 1, 'b': 2}

# Union: Either one type or another.
# Example: Union[int, str] - means the value can be an int or a str.

# Optional: Means a value can be the type you expect or None.
# Example: Optional[int] is the same as Union[int, None]

# Any: Anything at all - no restriction on type.
# You can pass an int, string, list, object etc.
from typing import List, Dict, Union, Optional, Any
from IPython.display import display, Markdown

print("LIbraries imported successfully...")
# Load environment variables from the .env file
load_dotenv()

# fetch API keys from environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")
# google_api_key = os.getenv("GOOGLE_API_KEY")
ollama_api_key = os.getenv("OLLAMA_API_KEY")

# Configure the clients
ollama_base_url = 'http://localhost:11434/v1'

openai_client = OpenAI(api_key= openai_api_key)
ollama_client = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)
# genai.configure(api_key = google_api_key)

# Initialize the Gemini model, choose a suitable model like "gemini-2.0-flash"
# gemini_model = genai.GenerativeModel("gemini-2.0-flash")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
LIbraries imported successfully...


In [11]:
# Let' define a Pydantic model called scientist that describes what a valid response should look like.
class scientist(BaseModel):
    name: str
    field : str
    known_for : list[str]
    birth_year: int

In [12]:
# Let's define a prompt
prompt = """
Give me a JSON object with details about a famous scientist.
Include the following fields: name, field, knwon_for, and birth_year.
"""

# Let's make the API call to OpenAI
# Note that we used "opena_client.beta.chat.completions.psarse()" since we want to parse it into structured output instead of just plain text.
response = openai_client.beta.chat.completions.parse(model = 'gpt-4o-mini',
                                                     messages = [{'role': 'user', 'content': prompt}],
                                                     temperature= 0,
                                                     response_format = scientist,
                                                     max_tokens = 300)

In [13]:
# print the parsed output
response.choices[0].message.content

'{"name":"Albert Einstein","field":"Physics","known_for":["Theory of Relativity","Photoelectric Effect","Mass-energy equivalence (E=mc²)"],"birth_year":1879}'

In [14]:
# Use json.loads() to turn the Json into a Python dictionary
json.loads(response.choices[0].message.content)

{'name': 'Albert Einstein',
 'field': 'Physics',
 'known_for': ['Theory of Relativity',
  'Photoelectric Effect',
  'Mass-energy equivalence (E=mc²)'],
 'birth_year': 1879}

In [16]:
# Now we'll use Ollama model.
response = ollama_client.beta.chat.completions.parse(model = 'gemma4:e4b',
                                                     messages=[{'role': 'user',
                                                                'content': prompt}],
                                                                temperature= 0,
                                                                response_format=scientist,
                                                                max_tokens = 1000)

In [17]:
json.loads(response.choices[0].message.content)

{'name': 'Albert Einstein',
 'field': 'Theoretical Physics',
 'known_for': ['Developing the theory of relativity (both special and general)',
  'The mass-energy equivalence formula (E=mc²)'],
 'birth_year': 1879}

In [22]:
class Destinations(BaseModel):
    city: str
    country: str
    top_attractions: list[str]

In [23]:
travel_prompt = """
Being a Tour Operator, your job is to suggest a destination and coutnry alongwith its top_attractions. return the 
output with fields city (str), country (str) and top_attractions (list[str])
"""

In [24]:
# Now we'll use Ollama model.
response = ollama_client.beta.chat.completions.parse(model = 'gemma4:e4b',
                                                     messages=[{'role': 'user',
                                                                'content': prompt}],
                                                                temperature= 0,
                                                                response_format=Destinations,
                                                                max_tokens = 1000)

In [25]:
json.loads(response.choices[0].message.content)

{'city': 'JSON',
 'country': 'USA',
 'top_attractions': ['Statue of Liberty', 'Times Square', 'Central Park']}

## Define the LLM Inputs Including Resume and Target Job Description.  
- Now that we covered the basics of Pydantic, let's start with our main project.
- We need to get the user's resume and the target job description.
- For this notebook, we'll start by defining these as multi-line strings in Python.  
Later in the course, we'll adapt this to take input or read from files.

In [26]:
# Helper function to display markdown nicely
def print_markdown(text):
    display(Markdown(text))
    

In [27]:
# Let's define a sample resume text
resume_text = """
JOHN ALEXANDER SMITH

Data Scientist | Machine Learning Engineer | AI Solutions Specialist

Email: [john.smith@email.com](mailto:john.smith@email.com)
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: [www.johnsmithai.com](http://www.johnsmithai.com)

---

## PROFESSIONAL SUMMARY

Results-driven Data Scientist with 6+ years of experience developing machine learning models, predictive analytics solutions, and AI-powered applications across finance, healthcare, and e-commerce domains. Skilled in Python, SQL, deep learning, natural language processing, and cloud-based machine learning platforms. Proven ability to translate complex business requirements into scalable data-driven solutions that improve operational efficiency and business outcomes.

---

## CORE SKILLS

Programming Languages:

* Python
* SQL
* R
* Java

Machine Learning:

* Supervised Learning
* Unsupervised Learning
* Deep Learning
* Ensemble Methods
* Feature Engineering
* Model Optimization

AI & NLP:

* Large Language Models (LLMs)
* Retrieval-Augmented Generation (RAG)
* Prompt Engineering
* LangChain
* Hugging Face Transformers
* Vector Databases

Data Analysis & Visualization:

* Pandas
* NumPy
* Matplotlib
* Seaborn
* Plotly
* Power BI
* Tableau

Cloud & MLOps:

* AWS
* Azure
* Docker
* Kubernetes
* MLflow
* CI/CD Pipelines

Databases:

* PostgreSQL
* MySQL
* MongoDB
* Snowflake

---

## PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Responsibilities:

* Developed machine learning models for customer churn prediction, improving retention by 18%.
* Built NLP pipelines to automate document classification and information extraction.
* Designed end-to-end MLOps workflows using MLflow and Docker.
* Led a team of 4 data scientists on enterprise AI initiatives.
* Implemented Retrieval-Augmented Generation systems for internal knowledge management.

Achievements:

* Reduced model deployment time by 40%.
* Increased prediction accuracy from 81% to 92%.
* Generated estimated annual savings of $1.2M through predictive maintenance solutions.

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Responsibilities:

* Built recommendation engines using collaborative filtering techniques.
* Developed fraud detection models using XGBoost and Random Forest algorithms.
* Performed exploratory data analysis on multi-million record datasets.
* Created executive dashboards for business stakeholders.

Achievements:

* Improved fraud detection precision by 25%.
* Reduced reporting time by 70% through automation.
* Delivered 15+ production-grade machine learning solutions.

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

Responsibilities:

* Created data pipelines and ETL workflows.
* Generated weekly and monthly business intelligence reports.
* Assisted in predictive analytics projects.

Achievements:

* Automated manual reporting processes saving 20+ hours per week.
* Improved data quality through validation frameworks.

---

## PROJECTS

Enterprise RAG Knowledge Assistant

Technologies:
Python, LangChain, FAISS, OpenAI API, Hugging Face

Description:
Built an enterprise knowledge assistant capable of answering questions from internal company documentation using Retrieval-Augmented Generation.

Key Results:

* Reduced information retrieval time by 85%.
* Supported over 5,000 internal documents.

Customer Churn Prediction System

Technologies:
Python, Scikit-learn, XGBoost, Power BI

Description:
Developed a predictive model to identify customers likely to leave subscription services.

Results:

* Achieved 93%  classification accuracy.
* Improved customer retention strategies.

Medical Image Classification

Technologies:
PyTorch, CNNs, Transfer Learning

Description:
Built a deep learning solution for disease detection from medical images.

Results:

* Achieved 95% validation accuracy.

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate
* TensorFlow Developer Certificate
* Databricks Machine Learning Professional

---

## PUBLICATIONS

Smith, J., "Optimizing Customer Retention Using Machine Learning Models"
International Journal of Data Science, 2023

Smith, J., "Practical Applications of Retrieval-Augmented Generation in Enterprises"
AI Systems Review, 2024

---

## ACHIEVEMENTS

* Winner, Data Science Innovation Challenge 2023
* Speaker at AI & ML Summit 2024
* Top Performer Award, TechNova Analytics (2023)

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

Artificial Intelligence, Open Source Contributions, Data Visualization, Cloud Computing, Research and Innovation

"""

In [31]:
# Let's define a sample job description.
job_description_text = """
About Sailpeak

At Sailpeak, we are a Brussels-based consulting firm specializing in guiding Banking & Insurance firms through the complexities of digital transformation.

With expertise in Business Strategy, Data & AI, and Digital Transformation, Sailpeak helps clients navigate the ever-evolving landscape and reach new heights of success.

🤖 About The Role

We are looking for a Data Science Intern with 0–2 years of experience who is excited about AI, data, automation, and innovation.

This is not a “theoretical only” internship. You’ll work on real internal AI projects, support consultants on client initiatives, contribute to AI benchmarks and workflows, and help shape how Sailpeak leverages AI internally.

You will work part-time (2.5 days/week) in a fast-growing and entrepreneurial environment where ownership, curiosity, and initiative are highly valued.

If you enjoy dynamic environments where no two weeks look the same, you’ll fit right in.

📊 You will:

Contribute to internal AI and Data Science projects such as the Sailpeak AI Barometer. 
Support consultants on data and AI-related client projects. 
Research and benchmark AI tools, technologies, and market trends. 
Help design and build AI workflows, automations, and internal productivity solutions. 
Analyze datasets and extract actionable insights. 
Assist in developing prototypes, dashboards, and proof-of-concepts. 
Explore opportunities to integrate Generative AI into internal and client-facing processes. 
Collaborate with different teams to identify AI use cases and improvement opportunities. 
Document findings, methodologies, and recommendations clearly. 
Participate in brainstorming sessions around innovation, AI adoption, and digital transformation. 

📊 You bring to the table:

0–2 years of experience or internships in Data Science, AI, Analytics, Computer Science, Engineering, or related fields. 
A Bachelor’s or Master’s degree (ongoing or completed) in Data Science, Computer Science, Engineering, Mathematics, AI, or a related field. 
Basic knowledge of Python and data analysis libraries (Pandas, NumPy, etc.). 
Familiarity with AI concepts, machine learning, or Generative AI tools. 
Interest in AI workflows, automation tools, and emerging technologies. 
Strong analytical and problem-solving skills. 
A proactive personality with curiosity and ownership. 
Someone who enjoys learning quickly and experimenting with new ideas. 
Ability to work independently while collaborating closely with a team. 

🗣️ Language Skills:

Fluency in English. 
French or Dutch is a plus. 

🤹 Other Skills:

Comfortable working in a fast-paced and evolving environment. 
Strong communication and collaboration skills. 
Attention to detail and structured thinking. 
Passion for AI, innovation, and continuous improvement. 
Interest in consulting, digital transformation, and emerging technologies. 

🎁 What’s In It For You

 Internship compensation aligned with market standards. 
 Hands-on experience in a fast-growing consulting and AI-driven environment. 
 Ownership and autonomy from day one. 
 Exposure to AI strategy, data science, innovation, automation, and digital transformation projects. 
 Mentorship from experienced consultants and leadership with backgrounds from companies like Google, PayPal, Semetis, and Sia Partners. 
 A vibrant office culture in Brussels with regular events, knowledge-sharing sessions, and team activities. 

🧑‍🔬 The Process

Step 1: 30-minute introductory call with our Talent team to discuss your background, motivation, and expectations.

Step 2: Practical discussion around AI, problem-solving, and data-related use cases.

Step 3: Final conversation with one of the founders or team leads.

The Offer.

Are you ready to kick-start your career in Data Science and AI while contributing to innovative projects in a fast-growing consulting company?

Join Sailpeak and grow with us!
"""

In [29]:
# Let's display the origional resume
print_markdown("**--- Original Resume ---**")
print_markdown(resume_text)

**--- Original Resume ---**


JOHN ALEXANDER SMITH

Data Scientist | Machine Learning Engineer | AI Solutions Specialist

Email: [john.smith@email.com](mailto:john.smith@email.com)
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: [www.johnsmithai.com](http://www.johnsmithai.com)

---

## PROFESSIONAL SUMMARY

Results-driven Data Scientist with 6+ years of experience developing machine learning models, predictive analytics solutions, and AI-powered applications across finance, healthcare, and e-commerce domains. Skilled in Python, SQL, deep learning, natural language processing, and cloud-based machine learning platforms. Proven ability to translate complex business requirements into scalable data-driven solutions that improve operational efficiency and business outcomes.

---

## CORE SKILLS

Programming Languages:

* Python
* SQL
* R
* Java

Machine Learning:

* Supervised Learning
* Unsupervised Learning
* Deep Learning
* Ensemble Methods
* Feature Engineering
* Model Optimization

AI & NLP:

* Large Language Models (LLMs)
* Retrieval-Augmented Generation (RAG)
* Prompt Engineering
* LangChain
* Hugging Face Transformers
* Vector Databases

Data Analysis & Visualization:

* Pandas
* NumPy
* Matplotlib
* Seaborn
* Plotly
* Power BI
* Tableau

Cloud & MLOps:

* AWS
* Azure
* Docker
* Kubernetes
* MLflow
* CI/CD Pipelines

Databases:

* PostgreSQL
* MySQL
* MongoDB
* Snowflake

---

## PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Responsibilities:

* Developed machine learning models for customer churn prediction, improving retention by 18%.
* Built NLP pipelines to automate document classification and information extraction.
* Designed end-to-end MLOps workflows using MLflow and Docker.
* Led a team of 4 data scientists on enterprise AI initiatives.
* Implemented Retrieval-Augmented Generation systems for internal knowledge management.

Achievements:

* Reduced model deployment time by 40%.
* Increased prediction accuracy from 81% to 92%.
* Generated estimated annual savings of $1.2M through predictive maintenance solutions.

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Responsibilities:

* Built recommendation engines using collaborative filtering techniques.
* Developed fraud detection models using XGBoost and Random Forest algorithms.
* Performed exploratory data analysis on multi-million record datasets.
* Created executive dashboards for business stakeholders.

Achievements:

* Improved fraud detection precision by 25%.
* Reduced reporting time by 70% through automation.
* Delivered 15+ production-grade machine learning solutions.

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

Responsibilities:

* Created data pipelines and ETL workflows.
* Generated weekly and monthly business intelligence reports.
* Assisted in predictive analytics projects.

Achievements:

* Automated manual reporting processes saving 20+ hours per week.
* Improved data quality through validation frameworks.

---

## PROJECTS

Enterprise RAG Knowledge Assistant

Technologies:
Python, LangChain, FAISS, OpenAI API, Hugging Face

Description:
Built an enterprise knowledge assistant capable of answering questions from internal company documentation using Retrieval-Augmented Generation.

Key Results:

* Reduced information retrieval time by 85%.
* Supported over 5,000 internal documents.

Customer Churn Prediction System

Technologies:
Python, Scikit-learn, XGBoost, Power BI

Description:
Developed a predictive model to identify customers likely to leave subscription services.

Results:

* Achieved 93%  classification accuracy.
* Improved customer retention strategies.

Medical Image Classification

Technologies:
PyTorch, CNNs, Transfer Learning

Description:
Built a deep learning solution for disease detection from medical images.

Results:

* Achieved 95% validation accuracy.

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate
* TensorFlow Developer Certificate
* Databricks Machine Learning Professional

---

## PUBLICATIONS

Smith, J., "Optimizing Customer Retention Using Machine Learning Models"
International Journal of Data Science, 2023

Smith, J., "Practical Applications of Retrieval-Augmented Generation in Enterprises"
AI Systems Review, 2024

---

## ACHIEVEMENTS

* Winner, Data Science Innovation Challenge 2023
* Speaker at AI & ML Summit 2024
* Top Performer Award, TechNova Analytics (2023)

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

Artificial Intelligence, Open Source Contributions, Data Visualization, Cloud Computing, Research and Innovation



In [32]:
# Let's display the target job description
print_markdown("*--- Target Job Description ---*")
print_markdown(job_description_text)

*--- Target Job Description ---*


About Sailpeak

At Sailpeak, we are a Brussels-based consulting firm specializing in guiding Banking & Insurance firms through the complexities of digital transformation.

With expertise in Business Strategy, Data & AI, and Digital Transformation, Sailpeak helps clients navigate the ever-evolving landscape and reach new heights of success.

🤖 About The Role

We are looking for a Data Science Intern with 0–2 years of experience who is excited about AI, data, automation, and innovation.

This is not a “theoretical only” internship. You’ll work on real internal AI projects, support consultants on client initiatives, contribute to AI benchmarks and workflows, and help shape how Sailpeak leverages AI internally.

You will work part-time (2.5 days/week) in a fast-growing and entrepreneurial environment where ownership, curiosity, and initiative are highly valued.

If you enjoy dynamic environments where no two weeks look the same, you’ll fit right in.

📊 You will:

Contribute to internal AI and Data Science projects such as the Sailpeak AI Barometer. 
Support consultants on data and AI-related client projects. 
Research and benchmark AI tools, technologies, and market trends. 
Help design and build AI workflows, automations, and internal productivity solutions. 
Analyze datasets and extract actionable insights. 
Assist in developing prototypes, dashboards, and proof-of-concepts. 
Explore opportunities to integrate Generative AI into internal and client-facing processes. 
Collaborate with different teams to identify AI use cases and improvement opportunities. 
Document findings, methodologies, and recommendations clearly. 
Participate in brainstorming sessions around innovation, AI adoption, and digital transformation. 

📊 You bring to the table:

0–2 years of experience or internships in Data Science, AI, Analytics, Computer Science, Engineering, or related fields. 
A Bachelor’s or Master’s degree (ongoing or completed) in Data Science, Computer Science, Engineering, Mathematics, AI, or a related field. 
Basic knowledge of Python and data analysis libraries (Pandas, NumPy, etc.). 
Familiarity with AI concepts, machine learning, or Generative AI tools. 
Interest in AI workflows, automation tools, and emerging technologies. 
Strong analytical and problem-solving skills. 
A proactive personality with curiosity and ownership. 
Someone who enjoys learning quickly and experimenting with new ideas. 
Ability to work independently while collaborating closely with a team. 

🗣️ Language Skills:

Fluency in English. 
French or Dutch is a plus. 

🤹 Other Skills:

Comfortable working in a fast-paced and evolving environment. 
Strong communication and collaboration skills. 
Attention to detail and structured thinking. 
Passion for AI, innovation, and continuous improvement. 
Interest in consulting, digital transformation, and emerging technologies. 

🎁 What’s In It For You

 Internship compensation aligned with market standards. 
 Hands-on experience in a fast-growing consulting and AI-driven environment. 
 Ownership and autonomy from day one. 
 Exposure to AI strategy, data science, innovation, automation, and digital transformation projects. 
 Mentorship from experienced consultants and leadership with backgrounds from companies like Google, PayPal, Semetis, and Sia Partners. 
 A vibrant office culture in Brussels with regular events, knowledge-sharing sessions, and team activities. 

🧑‍🔬 The Process

Step 1: 30-minute introductory call with our Talent team to discuss your background, motivation, and expectations.

Step 2: Practical discussion around AI, problem-solving, and data-related use cases.

Step 3: Final conversation with one of the founders or team leads.

The Offer.

Are you ready to kick-start your career in Data Science and AI while contributing to innovative projects in a fast-growing consulting company?

Join Sailpeak and grow with us!


## Enhance The Resume with OpenAI  
Now that we have our resume and job description, let's use the OpenAI API to improve the resume to better match the job requirements. We'll make a call to the text generation API and ask it to enhance our resume.

In [36]:
def openai_generate(prompt: str,
                    model: str = "gpt-4o",
                    temperature: float = 0.7,
                    max_tokens: int = 1500,
                    response_format: Optional[dict] = None) -> str | dict:
    """
    Generate text using OpenAI API

    This function sends a prompt to OpenAI's API and returns the generated response.
    It supports both standard text generation and structured parsing with response_format.

    Args:
        prompt (str): The prompt to send to the model, i.e.: your instructions for the AI
        model (str): The OpenAI model to use (default: "gpt-4o")
        temperature (float): Controls randomness, where lower values make output more deterministic
        max_tokens (int): Maximum number of tokens to generate, which limits the response length
        response_format (dict): Optional format specification
        In simple terms, response_format is optional. If the user gives me a dictionary, cool! 
        If they don't give me anything, just assume it's None and keep going."

    Returns:
        str or dict: The generated text or parsed structured data, depending on response_format
    """

    
    try:
        # Standard text generation without a specific response format
        if not response_format:
            response = openai_client.chat.completions.create(
                model = model,
                messages = [
                    {"role": "system",
                     "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt}],
                temperature = temperature,
                max_tokens = max_tokens)
            
            # Extract just the text content from the response
            return response.choices[0].message.content
        
        # Structured response generation (e.g., JSON format)
        else:
            completion = openai_client.beta.chat.completions.parse(
                model = model,  # Make sure to use a model that supports parse
                messages = [
                    # Same system and user messages as above
                    {
                        "role": "system",
                        "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature = temperature,
                response_format = response_format)

            # Return the parsed structured output
            return completion.choices[0].message.parsed
            
    except Exception as e:
        # Error handling to prevent crashes
        return f"Error generating text: {e}"


In [37]:
prompt = f"""
Context:
You are a professional resume writer helping a candidate tailor their resume for a specific job opportunity. The resume and job description are provided below.

Instruction:
Enhance the resume to make it more impactful. Focus on:
- Highlighting relevant skills and achievements.
- Using strong action verbs and quantifiable results where possible.
- Rewriting vague or generic bullet points to be specific and results-driven.
- Emphasizing experience and skills most relevant to the job description.
- Reorganizing sections if necessary to better match the job requirements.

Resume:
{resume_text}

Output:
Provide a revised and improved version of the resume that is well-formatted. Only return the updated resume.
"""

In [38]:
# Get response from OpenAI API
openai_output = openai_generate(prompt, temperature=0.7)

print_markdown(f"#### OpenAI Response #### \n{openai_output}")

#### OpenAI Response #### 
```plaintext
JOHN ALEXANDER SMITH

Data Scientist | Machine Learning Engineer | AI Solutions Specialist

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

## PROFESSIONAL SUMMARY

Innovative Data Scientist with over 6 years of experience in crafting state-of-the-art machine learning models and AI-driven solutions across finance, healthcare, and e-commerce sectors. Expertise in Python, SQL, deep learning, and cloud-based platforms with a track record of translating complex business needs into scalable, data-driven solutions that enhance business performance and efficiency.

---

## CORE SKILLS

**Programming Languages:**
- Python, SQL, R, Java

**Machine Learning:**
- Supervised & Unsupervised Learning, Deep Learning, Ensemble Methods, Feature Engineering, Model Optimization

**AI & NLP:**
- Large Language Models, Retrieval-Augmented Generation, Prompt Engineering, LangChain, Hugging Face Transformers, Vector Databases

**Data Analysis & Visualization:**
- Pandas, NumPy, Matplotlib, Seaborn, Plotly, Power BI, Tableau

**Cloud & MLOps:**
- AWS, Azure, Docker, Kubernetes, MLflow, CI/CD Pipelines

**Databases:**
- PostgreSQL, MySQL, MongoDB, Snowflake

---

## PROFESSIONAL EXPERIENCE

**Senior Data Scientist**  
TechNova Analytics | Austin, Texas  
January 2022 - Present

- Spearheaded the development of machine learning models for customer churn prediction, boosting retention by 18%.
- Engineered NLP pipelines to automate document classification, cutting processing time by 50%.
- Pioneered end-to-end MLOps workflows using MLflow and Docker, slashing model deployment time by 40%.
- Directed a team of 4 data scientists, leading enterprise AI initiatives and enhancing prediction accuracy from 81% to 92%.
- Implemented Retrieval-Augmented Generation systems, reducing knowledge retrieval time by 85%.

**Achievements:**
- Generated annual savings of $1.2M via predictive maintenance solutions.
- Recognized as Top Performer Award, TechNova Analytics (2023).

**Data Scientist**  
Insight Data Solutions | Dallas, Texas  
June 2019 - December 2021

- Constructed recommendation engines using collaborative filtering, enhancing user engagement.
- Developed fraud detection models using XGBoost and Random Forest, improving detection precision by 25%.
- Automated reporting processes, reducing reporting time by 70% and delivering 15+ production-grade solutions.

**Junior Data Analyst**  
SmartMetrics Inc. | Houston, Texas  
July 2017 - May 2019

- Designed data pipelines and ETL workflows, automating manual reporting processes to save 20+ hours weekly.
- Assisted in predictive analytics projects, improving data quality with validation frameworks.

---

## PROJECTS

**Enterprise RAG Knowledge Assistant**
- **Technologies:** Python, LangChain, FAISS, OpenAI API, Hugging Face
- **Results:** Reduced information retrieval time by 85%, supporting over 5,000 internal documents.

**Customer Churn Prediction System**
- **Technologies:** Python, Scikit-learn, XGBoost, Power BI
- **Results:** Achieved 93% classification accuracy, enhancing customer retention strategies.

**Medical Image Classification**
- **Technologies:** PyTorch, CNNs, Transfer Learning
- **Results:** Achieved 95% validation accuracy for disease detection.

---

## EDUCATION

**Master of Science (M.S.) in Data Science**  
University of Texas at Austin, 2017

**Bachelor of Science (B.S.) in Computer Science**  
Texas A&M University, 2015

---

## CERTIFICATIONS

- AWS Certified Machine Learning Specialty
- Microsoft Azure Data Scientist Associate
- TensorFlow Developer Certificate
- Databricks Machine Learning Professional

---

## PUBLICATIONS

- "Optimizing Customer Retention Using Machine Learning Models", International Journal of Data Science, 2023
- "Practical Applications of Retrieval-Augmented Generation in Enterprises", AI Systems Review, 2024

---

## ACHIEVEMENTS

- Winner, Data Science Innovation Challenge 2023
- Speaker at AI & ML Summit 2024

---

## LANGUAGES

- English (Native)
- Spanish (Professional Working Proficiency)

---

## INTERESTS

Artificial Intelligence, Open Source Contributions, Data Visualization, Cloud Computing, Research and Innovation
```

### Using Ollama and Model Gemma4:e4b

In [39]:
def ollama_generate(prompt: str,
                    model: str = "gemma4:e4b",
                    temperature: float = 0.7,
                    max_tokens: int = 1500,
                    response_format: Optional[dict] = None) -> str | dict:
    """
    Generate text using OpenAI API

    This function sends a prompt to OpenAI's API and returns the generated response.
    It supports both standard text generation and structured parsing with response_format.

    Args:
        prompt (str): The prompt to send to the model, i.e.: your instructions for the AI
        model (str): The OpenAI model to use (default: "gpt-4o")
        temperature (float): Controls randomness, where lower values make output more deterministic
        max_tokens (int): Maximum number of tokens to generate, which limits the response length
        response_format (dict): Optional format specification
        In simple terms, response_format is optional. If the user gives me a dictionary, cool! 
        If they don't give me anything, just assume it's None and keep going."

    Returns:
        str or dict: The generated text or parsed structured data, depending on response_format
    """

    
    try:
        # Standard text generation without a specific response format
        if not response_format:
            response = ollama_client.chat.completions.create(
                model = model,
                messages = [
                    {"role": "system",
                     "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt}],
                temperature = temperature,
                max_tokens = max_tokens)
            
            # Extract just the text content from the response
            return response.choices[0].message.content
        
        # Structured response generation (e.g., JSON format)
        else:
            completion = ollama_client.beta.chat.completions.parse(
                model = model,  # Make sure to use a model that supports parse
                messages = [
                    # Same system and user messages as above
                    {
                        "role": "system",
                        "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature = temperature,
                response_format = response_format)

            # Return the parsed structured output
            return completion.choices[0].message.parsed
            
    except Exception as e:
        # Error handling to prevent crashes
        return f"Error generating text: {e}"


In [40]:
# Same enhancement prompt as in previous function for OpenAI
prompt = f"""
Context:
You are a professional resume writer helping a candidate tailor their resume for a specific job opportunity. The resume and job description are provided below.

Instruction:
Enhance the resume to make it more impactful. Focus on:
- Highlighting relevant skills and achievements.
- Using strong action verbs and quantifiable results where possible.
- Rewriting vague or generic bullet points to be specific and results-driven.
- Emphasizing experience and skills most relevant to the job description.
- Reorganizing sections if necessary to better match the job requirements.

Resume:
{resume_text}

Output:
Provide a revised and improved version of the resume that is well-formatted. Only return the updated resume.
"""

In [41]:
# Get response from Ollama API
ollama_output = ollama_generate(prompt, temperature=0.7)

print_markdown(f"#### Gemma4 Response #### \n{ollama_output}")

#### OpenAI Response #### 
# JOHN ALEXANDER SMITH
**Data Scientist | Machine Learning Engineer | Applied AI Solutions Specialist**

Email: john.smith@email.com | Phone: +1 (555) 123-4567 | Austin, Texas, USA
[linkedin.com/in/johnsmith](http://www.linkedin.com/in/johnsmith/) | [github.com/johnsmith](http://www.github.com/johnsmith/) | [www.johnsmithai.com](http://www.johnsmithai.com)

---

## PROFESSIONAL SUMMARY
Highly accomplished and results-driven Data Scientist with 6+ years of experience specializing in the development, deployment, and scaling of advanced AI solutions, particularly in Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), and MLOps infrastructure. Proven expertise translating complex business challenges into high-impact, data-driven strategies across finance, healthcare, and e-commerce sectors. Adept at leading cross-functional teams to deliver production-grade ML systems that significantly boost operational efficiency and generate measurable revenue improvements.

---

## TECHNICAL EXPERTISE
**AI/ML Frameworks:** Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Prompt Engineering, Deep Learning (CNNs, RNNs), NLP Pipelines, Transformers, LangChain, Hugging Face.
**Programming & Data:** Python (Pandas, NumPy, Scikit-learn), SQL, R, Java.
**MLOps & Cloud:** AWS (SageMaker), Azure ML, Docker, Kubernetes, MLflow, CI/CD Pipelines, Version Control.
**Databases:** Snowflake, PostgreSQL, MySQL, MongoDB, Vector Databases (e.g., FAISS).
**Visualization:** Power BI, Tableau, Matplotlib, Seaborn, Plotly.

---

## PROFESSIONAL EXPERIENCE

**Senior Data Scientist** | TechNova Analytics | Austin, Texas
*January 2022 – Present*

Spearheaded the design and implementation of enterprise-level AI solutions, leading initiatives that directly impacted client revenue and operational efficiency.

*   **Generative AI & NLP:** Architected and deployed Retrieval-Augmented Generation (RAG) systems using LangChain and vector databases for internal knowledge management, significantly reducing information retrieval time by **85%** across over 5,000 proprietary documents.
*   **MLOps Leadership:** Designed and executed end-to-end MLOps workflows utilizing MLflow, Docker, and CI/CD pipelines, resulting in a **40% reduction** in model deployment cycle time and ensuring robust production scalability.
*   **Predictive Modeling:** Developed and scaled advanced machine learning models for customer churn prediction, improving client retention strategies by **18%**.
*   **Business Impact:** Generated an estimated annual savings of **$1.2 Million** through the implementation of predictive maintenance solutions across industrial clients.
*   Managed and mentored a cross-functional team of 4 data scientists on high-priority enterprise AI initiatives, driving best practices in model governance and performance optimization.

**Data Scientist** | Insight Data Solutions | Dallas, Texas
*June 2019 – December 2021*

Focused on developing robust predictive models for financial services and e-commerce platforms.

*   **Fraud Detection:** Engineered sophisticated fraud detection models using XGBoost and Random Forest algorithms, increasing overall detection precision by **25%** and minimizing financial losses.
*   **Recommendation Systems:** Built and optimized collaborative filtering recommendation engines that enhanced user engagement and boosted cross-sell opportunities for e-commerce clients.
*   **Data Analysis & Reporting:** Streamlined complex data analysis processes, automating manual reporting workflows and reducing client reporting time by **70%**.
*   Delivered over 15 production-grade machine learning solutions, handling multi-million record datasets and ensuring model reliability in live environments.

**Junior Data Analyst** | SmartMetrics Inc. | Houston, Texas
*July 2017 – May 2019*

Gained foundational experience in data infrastructure and business intelligence reporting.

*   Developed and maintained scalable ETL pipelines and data validation frameworks, improving overall data quality for executive decision-making.
*   Automated critical manual reporting processes, saving over **20+ hours** of manual labor per week for the operations team.
*   Assisted senior analysts in key predictive analytics projects, contributing foundational insights into market trends and operational bottlenecks.

---

## SELECTED PROJECTS & PORTFOLIO WORK

**Enterprise RAG Knowledge Assistant** (Portfolio Project)
*Technologies: Python, LangChain, FAISS, OpenAI API, Hugging Face*
Developed a comprehensive knowledge assistant capable of synthesizing answers from vast internal documentation using advanced Retrieval-Augmented Generation architecture. **Key Result:** Demonstrated ability to process and synthesize information from 5,000+ documents with an efficiency gain of 85%.

**Medical Image Classification System** (Deep Learning Project)
*Technologies: PyTorch, CNNs, Transfer Learning*
Built a deep learning solution for classifying medical images (e.g., disease detection). **Key Result:** Achieved a high validation accuracy rate of 95%, demonstrating proficiency in handling complex, sensitive datasets.

---

## EDUCATION & CERTIFICATIONS

**Master of Science (M.S.) in Data Science**
University of Texas at Austin | *2017*

**Bachelor of Science (B.S.) in Computer Science**
Texas A&M University | *2015*

**Certifications:** AWS Certified Machine

## Perform a Gap Analysis Between Resume and Job Description.  
Now let's use an LLM to analyze the resume and job description. We want the Ai to identify:
1. Key skills/requirements mentioned in the Job Description.
2. Relevant skills/experience present in the Resume.
3. Crucially, the mismatches or gaps - what the job asks for that the resume doesn't highlight well.  
We'll use OpenAI and Ollam (Gemma4) for this analysis Step. We need to craft a clear prompt.

In [42]:
# prompt to analyze the resume against the job description.
def analyze_resume_against_job_description(job_description_text: str, resume_text: str, model: str ) -> str:
    """
    Analyze the resume against the job description and return a structured comparison.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        model (str): The model to use for analysis ("openai" or "gemini").

    Returns:
        str: A clear, structured comparison of the resume and job description.
    """
    # This prompt instructs the AI to act as a career advisor and analyze how well the resume matches the job description
    # It asks for a structured analysis with 4 specific sections: requirements, matches, gaps, and strengths
    prompt = f"""
    Context:
    You are a career advisor and resume expert. Your task is to analyze a candidate's resume against a specific job description to assess alignment and identify areas for improvement.

    Instruction:
    Review the provided Job Description and Resume. Identify key skills, experiences, and qualifications in the Job Description and compare them to what's present in the Resume. Provide a structured analysis with the following sections:
    1. **Key Requirements from Job Description:** List the main skills, experiences, and qualifications sought by the employer.
    2. **Relevant Experience in Resume:** List the skills and experiences from the resume that match or align closely with the job requirements.
    3. **Gaps/Mismatches:** Identify important skills or qualifications from the Job Description that are missing, unclear, or underrepresented in the Resume.
    4. **Potential Strengths:** Highlight any valuable skills, experiences, or accomplishments in the resume that are not explicitly requested in the job description but could strengthen the application.

    Job Description:

    {job_description_text}

    Resume:

    {resume_text}

    Output:
    Return a clear, structured comparison with the four sections outlined above.
    """

    # This conditional block selects which AI model to use based onthe 'model parameter' in function call
    if model == 'openai':
        gap_analysis = openai_generate(prompt, temperature=0.5)
    elif model == 'ollama':
        gap_analysis = ollama_generate(prompt, temperature=0.5)
    else: 
        raise ValueError(f"Invalid model {model} selection")
    
    return gap_analysis

In [43]:
# Call the function to analyze the resume against the job description using a model
gap_analysis_openai = analyze_resume_against_job_description(job_description_text,
                                                             resume_text,
                                                             model = 'openai')

print_markdown(f"### OpenAI Model Response ### \n{gap_analysis_openai}")

### OpenAI Model Response ### 
### 1. Key Requirements from Job Description

- **Experience & Education:**
  - 0–2 years of experience or internships in Data Science, AI, Analytics, Computer Science, Engineering, or related fields.
  - Bachelor’s or Master’s degree (ongoing or completed) in Data Science, Computer Science, Engineering, Mathematics, AI, or a related field.

- **Technical Skills:**
  - Basic knowledge of Python and data analysis libraries (Pandas, NumPy, etc.).
  - Familiarity with AI concepts, machine learning, or Generative AI tools.
  - Ability to analyze datasets and extract actionable insights.
  - Experience in developing prototypes, dashboards, and proof-of-concepts.
  - Interest in AI workflows, automation tools, and emerging technologies.

- **Soft Skills:**
  - Strong analytical and problem-solving skills.
  - Proactive personality with curiosity and ownership.
  - Ability to work independently while collaborating closely with a team.
  - Strong communication and collaboration skills.
  - Attention to detail and structured thinking.
  - Passion for AI, innovation, and continuous improvement.

- **Language Skills:**
  - Fluency in English.
  - French or Dutch is a plus.

- **Other:**
  - Comfortable working in a fast-paced and evolving environment.
  - Interest in consulting, digital transformation, and emerging technologies.

### 2. Relevant Experience in Resume

- **Experience & Education:**
  - M.S. in Data Science and B.S. in Computer Science.
  - 6+ years of experience in data science and machine learning, which exceeds the requirement but shows extensive experience.

- **Technical Skills:**
  - Proficient in Python and data analysis libraries (Pandas, NumPy).
  - Experience with machine learning, AI concepts, and Generative AI tools such as Hugging Face Transformers.
  - Developed machine learning models and AI-powered applications.
  - Experience with data analysis and visualization tools (Matplotlib, Seaborn, Plotly, Power BI).
  - Experience in building prototypes and dashboards.

- **Soft Skills:**
  - Proven problem-solving skills with achievements in improving operational efficiency.
  - Experience leading teams and collaborating on projects.
  - Strong communication skills demonstrated through publications and speaking engagements.

- **Language Skills:**
  - Fluency in English.

### 3. Gaps/Mismatches

- **Experience & Education:**
  - The candidate has significantly more experience than required (6+ years vs. 0–2 years), which might be beyond the scope of an internship role.

- **Language Skills:**
  - No mention of French or Dutch proficiency, which is listed as a plus.

- **Other:**
  - The resume does not specifically mention an interest in consulting, although it is implied through the work experience in various domains.

### 4. Potential Strengths

- **Technical Expertise:**
  - Extensive experience in machine learning, AI, and data science with a strong track record of successful projects.
  - Advanced skills in cloud computing and MLOps, which could be beneficial for the role.

- **Leadership and Innovation:**
  - Demonstrated leadership experience in managing teams and leading AI initiatives.
  - Recognized achievements in innovation, including awards and publications.

- **Public Engagement:**
  - Experience as a speaker at industry summits and contributor to AI publications, indicating strong communication and thought leadership skills.

- **Additional Skills:**
  - Proficiency in additional programming languages (R, Java) and databases, offering a broader technical skill set.

This structured analysis highlights how the candidate's extensive experience and skills align with the job description and identifies areas where they exceed or differ from the requirements.

In [44]:
# Call the function to analyze the resume against the job description using a model
gap_analysis_ollama = analyze_resume_against_job_description(job_description_text,
                                                             resume_text,
                                                             model = 'ollama')

print_markdown(f"### Gemma Model Response ### \n{gap_analysis_ollama}")

### Gemma Model Response ### 
As an experienced career advisor and resume expert, I have analyzed your profile against the requirements of the Data Science Intern role at Sailpeak.

Overall, you possess a significantly deeper and more senior skill set than required by this internship. This is not a weakness, but it means your application narrative must be carefully adjusted to demonstrate enthusiasm for learning, collaboration, and supporting consultants—rather than simply leading projects.

Here is the structured analysis:

***

### 1. Key Requirements from Job Description (JD)
The employer is looking for a highly motivated individual who can contribute hands-on technical support within a consulting framework, focusing on emerging technologies.

**A. Core Technical Skills & Knowledge:**
*   Basic knowledge of Python and data analysis libraries (Pandas, NumPy).
*   Familiarity with AI concepts, machine learning, or Generative AI tools (LLMs, RAG).
*   Ability to develop prototypes, dashboards, and proof-of-concepts.

**B. Experience & Contribution Focus:**
*   Supporting consultants on data and AI-related client projects.
*   Contributing to internal AI/Data Science projects (e.g., benchmarks, internal tools).
*   Researching and benchmarking AI tools/market trends.
*   Designing workflows and automations.

**C. Soft Skills & Mindset:**
*   Proactive attitude, curiosity, and ownership ("highly valued").
*   Ability to learn quickly and experiment with new ideas.
*   Strong collaboration skills (working independently while collaborating closely).
*   Interest in consulting, digital transformation, and continuous improvement.

### 2. Relevant Experience in Resume (R)
Your resume demonstrates exceptional technical depth that aligns perfectly with the *technical scope* of the role, even if your experience level is higher than advertised.

**A. Direct Technical Alignment:**
*   **Python/Data Libraries:** Explicitly listed and used across all roles (Pandas, NumPy).
*   **ML & AI Concepts:** Deep expertise in Supervised Learning, Deep Learning, NLP, Feature Engineering, etc., which exceeds the "basic knowledge" requirement.
*   **Generative AI / LLMs:** Excellent, current experience with Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), LangChain, and Hugging Face Transformers—directly matching the JD's focus on GenAI integration.
*   **Prototypes & Dashboards:** Experience creating "executive dashboards" and multiple structured projects (e.g., Churn Prediction System).

**B. Workflow & Impact Alignment:**
*   **Automation/Efficiency:** Repeatedly achieving quantifiable improvements ("Reduced model deployment time by 40%", "Automated manual reporting processes saving 20+ hours per week"). This shows practical workflow improvement skills.
*   **Structured Thinking:** The use of MLOps (MLflow, Docker) and CI/CD pipelines demonstrates highly structured, production-grade thinking—a key asset in a consulting environment.

### 3. Gaps/Mismatches (Areas for Improvement)
The primary gaps are not technical skill deficits but **contextual alignment** and **framing**. Your experience is too senior for an

## Draft a New Tailored Resume By AI With Change Tracking (With Pydantic)  
Now, we'll use the insights gained (analysis and suggestions) to generate a completly rewritten, tailored resume using OpenAI. A key enhancement here is to ask the AI not just to list the changes, but to try and identify which sections or areas of the resume were modified. This helps the user quickly see the impact of the tailoring.  
We'll ask the AI for two outputs:
1. The full text of the Tailored resume.
2. A structured list (using Markdown) describing the key changes and where they were made (e.g., Summary, Experience section, Skills).

In [47]:
# Define Pydantic models for strucured output
# The ResumeOutput class is a pydantic model that defines the strucutre of the output
# for the resume generation function. It includes two fields:
#   (1). updated_resume: A string that contains the final rewritten resume.
#   (2). diff_markdown: A string containing the resume's HTML-coloured version highlighting additions and deletions.

class ResumeOutput(BaseModel):
    updated_resume: str
    diff_markdown: str

def generate_resume(
            job_description_text: str, resume_text: str, gap_analysis_openai: str, model : str = 'openai'
    ) -> str:
        """
    Generate a tailored resume using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        gap_analysis_openai (str): The gap analysis result from OpenAI.
        model (str): The model to use for resume generation.

    Returns:
        dict: A dictionary containing the updated resume and diff markdown.
    """
    # Construct the prompt for the AI model to generate the tailored resume.
    # The prompt includes context, instructions, and input data (original resume,
    # target job description, and gap analysis).
        prompt = (
        """
    ### Context:
    You are an expert resume writer and editor. Your goal is to rewrite the original resume to match the target job description, using the provided tailoring suggestions and analysis.

    ---

    ### Instruction:
    1. Rewrite the entire resume to best match the **Target Job Description** and **Gap Analysis to the Job Description**.
    2. Improve clarity, add job-relevant keywords, and quantify achievements.
    3. Specifically address the gaps identified in the analysis by:
       - Adding missing skills and technologies mentioned in the job description
       - Reframing experience to highlight relevant accomplishments
       - Strengthening sections that were identified as weak in the analysis
    4. Prioritize addressing the most critical gaps first
    5. Incorporate industry-specific terminology from the job description
    6. Ensure all quantifiable achievements are properly highlighted with metrics
    7. Return two versions of the resume:
        - `updated_resume`: The final rewritten resume (as plain text)
        - `diff_html`: A version of the resume with inline highlights using color:
            - Additions or rewritten content should be **green**:  
            `<span style="color:green">your added or changed text</span>`
            - Removed content should be **red and struck through**:  
            `<span style="color:red;text-decoration:line-through">removed text</span>`
            - Leave unchanged lines unmarked.
        - Keep all section headers and formatting consistent with the original resume.

    ---

    ### Output Format:

    ```json
    {
    "updated_resume": "<full rewritten resume as plain text>",
    "diff_markdown": "<HTML-colored version of the resume highlighting additions and deletions>"
    }
    ```
    ---
    ### Input:

    **Original Resume:**

    """
        + resume_text
        + """


    **Target Job Description:**

    """
        + job_description_text
        + """


    **Analysis of Resume vs. Job Description:**

    """
        + gap_analysis_openai
    )

    # Depending on the selected model, call the appropriate function to generate 
    # If the OpenAI model is selected, it uses a temperature of 0.7 for creating
        if model == 'openai':
            updated_resume_json = openai_generate(prompt, temperature=0.7, response_format=ResumeOutput)
        elif model == 'ollama':
            updated_resume_json = ollama_generate(prompt, temperature= 0.5, response_format=ResumeOutput)
        else:
            raise ValueError(f'Invalid model: {model} seleted.')
    
        return updated_resume_json



In [48]:
# Call the generate_resume function with the provied job description, resume text, and gap analysis.
updated_reume_json = generate_resume(job_description_text,
                                     resume_text, 
                                     gap_analysis_openai,
                                     model = 'openai')

# Display the updated resume in Markdown format.
print_markdown(updated_reume_json.updated_resume)

JOHN ALEXANDER SMITH

Aspiring Data Science Intern | AI Enthusiast | Innovation Advocate

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

## PROFESSIONAL SUMMARY

Motivated and innovative data science professional with a strong foundation in machine learning, AI, and data analytics. Passionate about leveraging AI and data science to drive digital transformation and solve complex business problems. Excited about contributing to real-world AI projects and enhancing data-driven decision-making.

---

## CORE SKILLS

Programming & Data Analysis:

* Python, Pandas, NumPy
* SQL, R

AI & Machine Learning:

* Supervised & Unsupervised Learning
* Deep Learning, Generative AI
* NLP, LangChain, Hugging Face Transformers

Data Visualization & Prototyping:

* Matplotlib, Seaborn, Plotly
* Power BI, Tableau

Cloud Computing & MLOps:

* AWS, Azure
* Docker, Kubernetes
* MLflow

---

## PROFESSIONAL EXPERIENCE

Data Science Professional
TechNova Analytics | Austin, Texas
January 2022 - Present

* Developed and implemented machine learning models to enhance operational efficiency, achieving a 92% prediction accuracy.
* Automated document classification and information extraction using NLP techniques, reducing processing time by 85%.
* Designed AI workflows and end-to-end MLOps solutions to improve data deployment speed by 40%.
* Collaborated with cross-functional teams to drive AI innovation and digital transformation initiatives.

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

* Created predictive models and recommendation systems to support customer retention and fraud detection.
* Conducted data analysis and visualization to extract actionable insights and drive business improvements.
* Developed executive dashboards and prototypes to facilitate data-driven decision-making.

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

* Supported data analytics projects by generating reports and building ETL workflows.
* Assisted in developing automated solutions to improve data quality and reporting efficiency.

---

## PROJECTS

Enterprise AI Solutions

* Developed AI-powered applications to enhance information retrieval and customer engagement.
* Integrated Generative AI tools to create innovative solutions for real-world challenges.

Medical Image Classification

* Implemented deep learning models to achieve 95% accuracy in disease detection from medical images.

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate
* TensorFlow Developer Certificate

---

## PUBLICATIONS

* "Optimizing Customer Retention Using Machine Learning Models" - International Journal of Data Science, 2023
* "Practical Applications of Retrieval-Augmented Generation in Enterprises" - AI Systems Review, 2024

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

AI & Innovation, Digital Transformation, Data Visualization, Cloud Computing, Continuous Learning


In [49]:
print_markdown(updated_reume_json.diff_markdown)

JOHN ALEXANDER SMITH

<span style="color:red;text-decoration:line-through">Data Scientist | Machine Learning Engineer | AI Solutions Specialist</span>
<span style="color:green">Aspiring Data Science Intern | AI Enthusiast | Innovation Advocate</span>

Email: [john.smith@email.com](mailto:john.smith@email.com)
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: [www.johnsmithai.com](http://www.johnsmithai.com)

---

## PROFESSIONAL SUMMARY

<span style="color:red;text-decoration:line-through">Results-driven Data Scientist with 6+ years of experience developing machine learning models, predictive analytics solutions, and AI-powered applications across finance, healthcare, and e-commerce domains. Skilled in Python, SQL, deep learning, natural language processing, and cloud-based machine learning platforms. Proven ability to translate complex business requirements into scalable data-driven solutions that improve operational efficiency and business outcomes.</span>

<span style="color:green">Motivated and innovative data science professional with a strong foundation in machine learning, AI, and data analytics. Passionate about leveraging AI and data science to drive digital transformation and solve complex business problems. Excited about contributing to real-world AI projects and enhancing data-driven decision-making.</span>

---

## CORE SKILLS

Programming & Data Analysis:

* Python, Pandas, NumPy
* SQL, R

AI & Machine Learning:

* Supervised & Unsupervised Learning
* Deep Learning, Generative AI
* NLP, LangChain, Hugging Face Transformers

Data Visualization & Prototyping:

* Matplotlib, Seaborn, Plotly
* Power BI, Tableau

Cloud Computing & MLOps:

* AWS, Azure
* Docker, Kubernetes
* MLflow

---

## PROFESSIONAL EXPERIENCE

<span style="color:red;text-decoration:line-through">Senior Data Scientist</span>
<span style="color:green">Data Science Professional</span>
TechNova Analytics | Austin, Texas
January 2022 - Present

<span style="color:red;text-decoration:line-through">Responsibilities:</span>

* Developed <span style="color:red;text-decoration:line-through">machine learning models for customer churn prediction, improving retention by 18%</span><span style="color:green">and implemented machine learning models to enhance operational efficiency, achieving a 92% prediction accuracy</span>.
* <span style="color:red;text-decoration:line-through">Built</span><span style="color:green">Automated</span> NLP pipelines to automate document classification and information extraction<span style="color:green">, reducing processing time by 85%</span>.
* Designed <span style="color:red;text-decoration:line-through">end-to-end MLOps workflows using MLflow and Docker</span><span style="color:green">AI workflows and end-to-end MLOps solutions to improve data deployment speed by 40%</span>.
* <span style="color:red;text-decoration:line-through">Led a team of 4 data scientists on enterprise AI initiatives</span><span style="color:green">Collaborated with cross-functional teams to drive AI innovation and digital transformation initiatives</span>.
* <span style="color:red;text-decoration:line-through">Implemented Retrieval-Augmented Generation systems for internal knowledge management</span>

<span style="color:red;text-decoration:line-through">Achievements:</span>

* <span style="color:red;text-decoration:line-through">Reduced model deployment time by 40%</span>.
* <span style="color:red;text-decoration:line-through">Increased prediction accuracy from 81% to 92%</span>.
* <span style="color:red;text-decoration:line-through">Generated estimated annual savings of $1.2M through predictive maintenance solutions</span>.

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

* <span style="color:red;text-decoration:line-through">Built recommendation engines using collaborative filtering techniques</span><span style="color:green">Created predictive models and recommendation systems to support customer retention and fraud detection</span>.
* Developed fraud detection models using <span style="color:red;text-decoration:line-through">XGBoost and Random Forest algorithms</span>.
* Performed exploratory data analysis on multi-million record datasets.
* <span style="color:red;text-decoration:line-through">Created</span><span style="color:green">Conducted</span> <span style="color:green">data analysis and visualization to extract actionable insights and drive business improvements</span>.
* <span style="color:red;text-decoration:line-through">Created executive dashboards</span><span style="color:green">Developed executive dashboards and prototypes to facilitate data-driven decision-making</span>.

<span style="color:red;text-decoration:line-through">Achievements:</span>

* <span style="color:red;text-decoration:line-through">Improved fraud detection precision by 25%</span>.
* <span style="color:red;text-decoration:line-through">Reduced reporting time by 70% through automation</span>.
* <span style="color:red;text-decoration:line-through">Delivered 15+ production-grade machine learning solutions</span>.

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

* <span style="color:red;text-decoration:line-through">Created data pipelines and ETL workflows</span><span style="color:green">Supported data analytics projects by generating reports and building ETL workflows</span>.
* <span style="color:red;text-decoration:line-through">Generated weekly and monthly business intelligence reports</span>.
* Assisted in <span style="color:red;text-decoration:line-through">predictive analytics projects</span><span style="color:green">developing automated solutions to improve data quality and reporting efficiency</span>.

<span style="color:red;text-decoration:line-through">Achievements:</span>

* <span style="color:red;text-decoration:line-through">Automated manual reporting processes saving 20+ hours per week</span>.
* <span style="color:red;text-decoration:line-through">Improved data quality through validation frameworks</span>.

---

## PROJECTS

<span style="color:red;text-decoration:line-through">Enterprise RAG Knowledge Assistant</span>

<span style="color:red;text-decoration:line-through">Technologies:</span>
<span style="color:red;text-decoration:line-through">Python, LangChain, FAISS, OpenAI API, Hugging Face</span>

<span style="color:red;text-decoration:line-through">Description:</span>
<span style="color:red;text-decoration:line-through">Built an enterprise knowledge assistant capable of answering questions from internal company documentation using Retrieval-Augmented Generation.</span>

<span style="color:red;text-decoration:line-through">Key Results:</span>

<span style="color:red;text-decoration:line-through">* Reduced information retrieval time by 85%.</span>
<span style="color:red;text-decoration:line-through">* Supported over 5,000 internal documents.</span>

<span style="color:green">Enterprise AI Solutions</span>

<span style="color:green">* Developed AI-powered applications to enhance information retrieval and customer engagement.</span>
* <span style="color:green">Integrated Generative AI tools to create innovative solutions for real-world challenges.</span>

<span style="color:red;text-decoration:line-through">Customer Churn Prediction System</span>

<span style="color:red;text-decoration:line-through">Technologies:</span>
<span style="color:red;text-decoration:line-through">Python, Scikit-learn, XGBoost, Power BI</span>

<span style="color:red;text-decoration:line-through">Description:</span>
<span style="color:red;text-decoration:line-through">Developed a predictive model to identify customers likely to leave subscription services.</span>

<span style="color:red;text-decoration:line-through">Results:</span>

<span style="color:red;text-decoration:line-through">* Achieved 93% classification accuracy.</span>
<span style="color:red;text-decoration:line-through">* Improved customer retention strategies.</span>

Medical Image Classification

Technologies:
PyTorch, CNNs, Transfer Learning

Description:
Built a deep learning solution for disease detection from medical images.

Results:

* Achieved 95% validation accuracy.

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate
* TensorFlow Developer Certificate
<span style="color:red;text-decoration:line-through">* Databricks Machine Learning Professional</span>

---

## PUBLICATIONS

<span style="color:red;text-decoration:line-through">Smith, J., "Optimizing Customer Retention Using Machine Learning Models"
International Journal of Data Science, 2023</span>

<span style="color:red;text-decoration:line-through">Smith, J., "Practical Applications of Retrieval-Augmented Generation in Enterprises"
AI Systems Review, 2024</span>

* <span style="color:green">"Optimizing Customer Retention Using Machine Learning Models" - International Journal of Data Science, 2023</span>
* <span style="color:green">"Practical Applications of Retrieval-Augmented Generation in Enterprises" - AI Systems Review, 2024</span>

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

<span style="color:red;text-decoration:line-through">Artificial Intelligence, Open Source Contributions, Data Visualization, Cloud Computing, Research and Innovation</span>
<span style="color:green">AI & Innovation, Digital Transformation, Data Visualization, Cloud Computing, Continuous Learning</span>


In [50]:
# Call the generate_resume function with the provied job description, resume text, and gap analysis.
updated_reume_json = generate_resume(job_description_text,
                                     resume_text, 
                                     gap_analysis_ollama,
                                     model = 'ollama')

# Display the updated resume in Markdown format.
print_markdown(updated_reume_json.updated_resume)

JOHN ALEXANDER SMITH
Data Scientist | AI Solutions Specialist

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

PROFESSIONAL SUMMARY

A highly motivated and proactive Data Scientist with 6+ years of experience in developing AI-driven solutions, specializing in Generative AI (LLMs) and predictive analytics. Proven ability to translate complex business challenges into actionable data science prototypes and workflows. Passionate about digital transformation and consulting methodologies, eager to support client initiatives by researching emerging technologies, building proof-of-concepts, and automating processes within a fast-paced environment.

---

CORE SKILLS
Programming Languages: Python, SQL, R, Java

Machine Learning & AI:
* Generative AI (LLMs), Retrieval-Augmented Generation (RAG)
* Natural Language Processing (NLP), Prompt Engineering
* Supervised/Unsupervised Learning, Deep Learning, Ensemble Methods
* Feature Engineering, Model Optimization

Tools & Platforms: AWS, Azure, Docker, Kubernetes, MLflow, LangChain, Hugging Face Transformers

Data Analysis & Visualization:
* Pandas, NumPy, Matplotlib, Seaborn, Plotly
* Power BI, Tableau

Databases: PostgreSQL, MySQL, MongoDB, Snowflake

Consulting Focus Areas:
* AI Workflow Design and Automation
* Business Process Improvement (BPI)
* Data Benchmarking & Research
* Digital Transformation Strategy

---

PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Key Contributions:

* Designed and implemented end-to-end MLOps workflows using MLflow and Docker to ensure scalable model deployment.
* Developed advanced NLP pipelines for automated document classification and information extraction (supporting client use cases).
* Led AI initiatives, focusing on implementing Retrieval-Augmented Generation (RAG) systems for internal knowledge management, significantly improving data access efficiency.
* Contributed to the development of predictive models (e.g., customer churn), enhancing business metrics by 18% and generating estimated annual savings of $1.2M through predictive maintenance solutions.
* Collaborated cross-functionally with stakeholders on AI benchmarks and strategic use case identification.

---

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Key Contributions:

* Developed and optimized machine learning solutions (e.g., fraud detection using XGBoost) that improved precision by 25%.
* Designed and built recommendation engines utilizing collaborative filtering techniques for business growth.
* Performed comprehensive exploratory data analysis on multi-million record datasets, translating findings into actionable insights for executive stakeholders via dashboards.
* Automated core reporting processes, reducing manual reporting time by 70% and saving significant operational hours.

---

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

Key Contributions:

* Managed data pipelines and ETL workflows to ensure high data quality for business intelligence reporting.
* Generated regular business intelligence reports and assisted in initial predictive analytics projects, developing foundational skills in structured problem-solving.


In [52]:
print_markdown(updated_reume_json.diff_markdown)

JOHN ALEXANDER SMITH
Data Scientist | <span style="color:green">AI Solutions Specialist</span>

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

PROFESSIONAL SUMMARY

A highly motivated and proactive Data Scientist with 6+ years of experience in developing AI-driven solutions, specializing in Generative AI (LLMs) and predictive analytics. Proven ability to translate complex business challenges into actionable data science prototypes and workflows. Passionate about digital transformation and consulting methodologies, eager to support client initiatives by researching emerging technologies, building proof-of-concepts, and automating processes within a fast-paced environment.

---

CORE SKILLS
Programming Languages: Python, SQL, R, Java

Machine Learning & AI:
* Generative AI (LLMs), Retrieval-Augmented Generation (RAG)
* Natural Language Processing (NLP), Prompt Engineering
* Supervised/Unsupervised Learning, Deep Learning, Ensemble Methods
* Feature Engineering, Model Optimization

Tools & Platforms: AWS, Azure, Docker, Kubernetes, MLflow, LangChain, Hugging Face Transformers

Data Analysis & Visualization:
* Pandas, NumPy, Matplotlib, Seaborn, Plotly
* Power BI, Tableau

Databases: PostgreSQL, MySQL, MongoDB, Snowflake

Consulting Focus Areas:
* AI Workflow Design and Automation
* Business Process Improvement (BPI)
* Data Benchmarking & Research
* Digital Transformation Strategy

---

PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Key Contributions:

* Designed and implemented end-to-end MLOps workflows using MLflow and Docker to ensure scalable model deployment.
* Developed advanced NLP pipelines for automated document classification and information extraction (supporting client use cases).
* Led AI initiatives, focusing on implementing Retrieval-Augmented Generation (RAG) systems for internal knowledge management, significantly improving data access efficiency.
* Contributed to the development of predictive models (e.g., customer churn), enhancing business metrics by 18% and generating estimated annual savings of $1.2M through predictive maintenance solutions.
* Collaborated cross-functionally with stakeholders on AI benchmarks and strategic use case identification.

---

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Key Contributions:

* Developed and optimized machine learning solutions (e.g., fraud detection using XGBoost) that improved precision by 25%.
* Designed and built recommendation engines utilizing collaborative filtering techniques for business growth.
* Performed comprehensive exploratory data analysis on multi-million record datasets, translating findings into actionable insights for executive stakeholders via dashboards.
* Automated core reporting processes, reducing manual reporting time by 70% and saving significant operational hours.

---

Junior Data Analyst
SmartMetrics Inc. | Houston, Texas
July 2017 - May 2019

Key Contributions:

* Managed data pipelines and ETL workflows to ensure high data quality for business intelligence reporting.
* Generated regular business intelligence reports and assisted in initial predictive analytics projects, developing foundational skills in structured problem-solving.


## Generate A Custom Cover Letter  
With the newly tailored resume, let's generate a corresponding cover letter. We'll use OpenAI, feeding it the tailored resume (from the previous task) and the origional job description. This ensures the cover letter highlights the most relevant points from the improved resume.

In [55]:
# Define Pydantic models for structrued output
# The CoverLetterOutput class is a Pydantic model that defines the structure of the output for the cover letter generation.
# It ensures that the output will contain a single field, 'cover_letter', which is a string.

class CoverLetterOutput(BaseModel):
    cover_letter: str


# The generate_cover_letter function creates a cover letter based on the provided job description and updated resume.
# It takes three parameters:
# (1). job_description_text: A string containing the job description for the position.
# (2). updated_resume: A string containing the candidate's updated resume.
# (3). model: A string indicating which model to use for generating the cover letter (default is "openai")
# The functino returns a dictionary containing the generated cover letter.

def generate_cover_letter(job_description_text:str,
                          updated_resume: str, 
                          model: str = 'openai') -> dict:
    
    """
    Generate a cover letter using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        updated_resume (str): The candidate's updated resume text.
        model (str): The model to use for cover letter generation.

    Returns:
        dict: A dictionary containing the cover letter.
    """

    # Construct the prompt for the AI model, including context and instructions for writing the cover letter.
    prompt = (
        """
    ### Context:
    You are a professional career coach and expert cover letter writer.

    ---

    ### Instruction:
    Write a compelling, personalized cover letter based on the **Updated Resume** and the **Target Job Description**. The letter should:
    1. Be addressed generically (e.g., "Dear Hiring Manager")
    2. Be no longer than 4 paragraphs
    3. Highlight key achievements and experiences from the updated resume
    4. Align with the responsibilities and qualifications in the job description
    5. Reflect the applicant's enthusiasm and fit for the role
    6. End with a confident and polite closing statement

    ---

    ### Output Format (JSON):
    ```json
    {
    "cover_letter": "<final cover letter text>"
    }
    ```
    ---

    ### Input:

    **Updated Resume:**

    """
        + updated_resume
        + """
    **Target Job Description:**

    """
        + job_description_text
    )

    if model == 'openai':
        updated_cover_letter = openai_generate(prompt,
        temperature= 0.7,
        response_format=CoverLetterOutput)
    elif model =="ollama":
        updated_cover_letter = ollama_generate(prompt,
                                               temperature = 0.7,
                                               response_format=CoverLetterOutput)
    else:
        raise ValueError(f"Invalid model: {model}")
    
    return updated_cover_letter

In [56]:
# Call the generate_cover_letter function with the provided job description
updated_cover_letter = generate_cover_letter(job_description_text,
                                             updated_reume_json.updated_resume, model = 'openai')

print_markdown(updated_cover_letter.cover_letter)

Dear Hiring Manager,

I am writing to express my strong interest in the Data Science Intern position at Sailpeak, as advertised. With a robust background in AI-driven solutions and a passion for digital transformation, I am eager to contribute to your consulting firm’s innovative projects. My experiences in developing AI workflows and predictive models align closely with the dynamic and entrepreneurial environment at Sailpeak, where I am confident my skills will be a great match.

In my current role as a Senior Data Scientist at TechNova Analytics, I have honed my abilities in designing and implementing end-to-end MLOps workflows and developing advanced NLP pipelines. These experiences have equipped me with the technical proficiency and problem-solving skills necessary to support consultants on client initiatives and contribute to AI benchmarks and workflows at Sailpeak. My work on Retrieval-Augmented Generation systems has significantly improved data access efficiency, demonstrating my ability to drive innovation and enhance productivity solutions.

The opportunity to work on internal AI projects, such as the Sailpeak AI Barometer, excites me, as does the chance to research and benchmark AI tools and technologies. My familiarity with machine learning, Generative AI tools, and platforms like AWS and Azure will allow me to effectively support your data and AI-related projects. Additionally, my proactive nature and strong analytical skills will enable me to identify AI use cases and improvement opportunities, aligning perfectly with Sailpeak’s mission.

I am thrilled at the prospect of joining Sailpeak and contributing to your team’s success. I am eager to bring my expertise in AI and data science to your firm, helping to shape innovative solutions that meet your clients’ needs. Thank you for considering my application. I look forward to the opportunity to discuss how I can contribute to Sailpeak’s goals.

Sincerely,

John Alexander Smith

## Unified Resume And Cover Letter Generation Function.  
Now that we have all the building blocks, let's create a single function,
`run_resume_rocket`, that takes the origional resume and job description text and performs the entire workflow: gap_analysis, resume tailoring with diff tracking, and cover letter generation. This makes the tool much easier to reuse.

In [58]:
def run_resume_rocket(resume_text: str, job_description_text: str) -> tuple[str, str]:
    """
    Run the resume rocket workflow.

    Args:
        resume_text (str): The candidate's resume text.
        job_description_text (str): The job description text.

    Returns:
        tuple: A tuple containing the updated resume and cover letter.
    """
    # Analyze the candidate's resume against the job description using OpenAI's model.
    # This function will return a structured analysis highlighting gaps and strengths.
    gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                                 resume_text, 
                                                                 model="openai")

    # Display the gap analysis results in Markdown format for better readability.
    print_markdown(gap_analysis_openai)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Generate an updated resume based on the job description, original resume, and gap analysis.
    # This function will return a JSON-like object containing the updated resume and a diff markdown.
    updated_resume_json = generate_resume(job_description_text, 
                                          resume_text, 
                                          gap_analysis_openai, 
                                          model = "openai")

    # Display the diff markdown which shows the changes made to the resume.
    print_markdown(updated_resume_json.diff_markdown)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Display the updated resume in Markdown format.
    print_markdown(updated_resume_json.updated_resume)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Generate a cover letter based on the job description and the updated resume.
    # This function will return the generated cover letter.
    updated_cover_letter = generate_cover_letter(
        job_description_text, updated_resume_json.updated_resume, model="openai"
    )

    # Display the generated cover letter in Markdown format.
    print_markdown(updated_cover_letter.cover_letter)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Return the updated resume and the generated cover letter as a tuple.
    return updated_resume_json.updated_resume, updated_cover_letter.cover_letter



In [59]:
# Call the run_resume_rocket function with the provided resume and job description
resume, cover_letter = run_resume_rocket(resume_text, 
                                         job_description_text,
                                         )

### 1. Key Requirements from Job Description:

- **Experience & Education:**
  - 0–2 years of experience or internships in Data Science, AI, Analytics, Computer Science, Engineering, or related fields.
  - Bachelor’s or Master’s degree (ongoing or completed) in Data Science, Computer Science, Engineering, Mathematics, AI, or a related field.

- **Technical Skills:**
  - Basic knowledge of Python and data analysis libraries (Pandas, NumPy, etc.).
  - Familiarity with AI concepts, machine learning, or Generative AI tools.

- **Soft Skills & Attributes:**
  - Strong analytical and problem-solving skills.
  - Proactive personality with curiosity and ownership.
  - Ability to work independently while collaborating closely with a team.
  - Fluency in English; French or Dutch is a plus.
  - Strong communication and collaboration skills.
  - Attention to detail and structured thinking.
  - Passion for AI, innovation, and continuous improvement.

- **Role-Specific Responsibilities:**
  - Contribute to internal AI and Data Science projects.
  - Support consultants on data and AI-related client projects.
  - Research AI tools, technologies, and market trends.
  - Design and build AI workflows and automations.
  - Analyze datasets for actionable insights.
  - Develop prototypes, dashboards, and proof-of-concepts.
  - Explore opportunities to integrate Generative AI.
  - Document findings and methodologies.
  - Participate in brainstorming sessions on innovation and AI adoption.

### 2. Relevant Experience in Resume:

- **Experience & Education:**
  - Master of Science in Data Science and Bachelor of Science in Computer Science, which exceeds the educational requirements.
  - Over 6 years of experience in data science and AI, which is beyond the required 0–2 years.

- **Technical Skills:**
  - Proficient in Python and data analysis libraries (Pandas, NumPy).
  - Extensive experience with machine learning, AI concepts, and Generative AI tools (LangChain, Hugging Face Transformers).
  - Experience with data visualization tools (Matplotlib, Seaborn, Plotly, Power BI, Tableau).

- **Soft Skills & Attributes:**
  - Strong analytical and problem-solving skills demonstrated through projects and professional roles.
  - Experience leading teams and collaborating, indicating strong communication and collaboration skills.
  - Demonstrated passion for AI and innovation through interests and achievements.

- **Role-Specific Responsibilities:**
  - Experience developing machine learning models and AI-powered applications.
  - Built NLP pipelines and automated workflows, aligning with the role’s requirement for designing AI workflows and automations.
  - Developed prototypes and proof-of-concepts in professional projects.

### 3. Gaps/Mismatches:

- **Experience Level:** The candidate has significantly more experience (6+ years) than the internship requires (0–2 years), which might be seen as overqualified for an internship position.
- **Language Skills:** The resume does not mention proficiency in French or Dutch, which is listed as a plus in the job description.
- **Location:** The candidate is based in Austin, Texas, while the job is in Brussels, which might imply relocation or remote work considerations.

### 4. Potential Strengths:

- **Advanced Technical Expertise:** The candidate’s extensive experience in machine learning, AI, and cloud computing could provide a deeper level of insight and innovation to the team.
- **Leadership Experience:** Experience leading a team of data scientists could be leveraged in collaborative projects and mentoring less experienced team members.
- **Publications and Achievements:** Publications in reputable journals and awards indicate a high level of expertise and recognition in the field, which could bring additional credibility and thought leadership to the role.
- **Bilingual Skills:** Proficiency in Spanish, although not listed as a requirement, could be beneficial if the company has Spanish-speaking clients or projects.


--------------------------------
--------------------------------



JOHN ALEXANDER SMITH

Data Scientist | <span style="color:red;text-decoration:line-through">Machine Learning Engineer</span> <span style="color:green">AI Solutions Enthusiast</span> <span style="color:red;text-decoration:line-through">| AI Solutions Specialist</span> | Innovation Advocate

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

## PROFESSIONAL SUMMARY

<span style="color:red;text-decoration:line-through">Results-driven</span> <span style="color:green">Passionate</span> Data Scientist with <span style="color:red;text-decoration:line-through">6+</span> <span style="color:green">over 6</span> years of experience <span style="color:red;text-decoration:line-through">developing machine learning models, predictive analytics solutions, and AI-powered applications across finance, healthcare, and e-commerce domains</span>, <span style="color:green">seeking to leverage my expertise in AI and data science to contribute to innovative projects at Sailpeak</span>. Skilled in Python, <span style="color:red;text-decoration:line-through">SQL, deep learning, natural language processing, and cloud-based machine learning platforms</span> <span style="color:green">data analysis, and machine learning</span>, I excel in transforming complex data into actionable insights. <span style="color:red;text-decoration:line-through">Proven ability to translate complex business requirements into scalable data-driven solutions that improve operational efficiency and business outcomes.</span> <span style="color:green">Known for my proactive approach, strong analytical skills, and ability to collaborate effectively in dynamic environments. Committed to continuous improvement and digital transformation in the Banking & Insurance sectors.</span>

---

## CORE SKILLS

Programming Languages:

* Python
* SQL
* <span style="color:red;text-decoration:line-through">R</span>
<span style="color:red;text-decoration:line-through">* Java</span>

Machine Learning <span style="color:green">& AI</span>:

* Supervised Learning
* Unsupervised Learning
* <span style="color:red;text-decoration:line-through">Deep Learning</span>
<span style="color:green">* Generative AI Tools</span>
* <span style="color:red;text-decoration:line-through">Ensemble Methods</span>
* Feature Engineering
* Model Optimization

<span style="color:red;text-decoration:line-through">AI & NLP:</span>

<span style="color:red;text-decoration:line-through">* Large Language Models (LLMs)</span>
<span style="color:red;text-decoration:line-through">* Retrieval-Augmented Generation (RAG)</span>
<span style="color:red;text-decoration:line-through">* Prompt Engineering</span>
<span style="color:red;text-decoration:line-through">* LangChain</span>
<span style="color:red;text-decoration:line-through">* Hugging Face Transformers</span>
<span style="color:red;text-decoration:line-through">* Vector Databases</span>

Data Analysis & Visualization:

* Pandas
* NumPy
* Matplotlib
* Seaborn
<span style="color:red;text-decoration:line-through">* Plotly</span>
* Power BI
<span style="color:red;text-decoration:line-through">* Tableau</span>

Cloud & MLOps:

* AWS
* Azure
* Docker
* Kubernetes
<span style="color:red;text-decoration:line-through">* MLflow</span>
<span style="color:red;text-decoration:line-through">* CI/CD Pipelines</span>

<span style="color:red;text-decoration:line-through">Databases:</span>

<span style="color:red;text-decoration:line-through">* PostgreSQL</span>
<span style="color:red;text-decoration:line-through">* MySQL</span>
<span style="color:red;text-decoration:line-through">* MongoDB</span>
<span style="color:red;text-decoration:line-through">* Snowflake</span>

Soft Skills:

* Strong analytical and problem-solving abilities
* Proactive with a strong sense of ownership
* Excellent communication and collaboration skills

---

## PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Responsibilities:

* Developed machine learning models <span style="color:red;text-decoration:line-through">for customer churn prediction, improving retention by 18%</span> <span style="color:green">for various predictive analytics projects, contributing to improved business outcomes</span>.
<span style="color:red;text-decoration:line-through">* Built NLP pipelines to automate document classification and information extraction.</span>
* <span style="color:green">Created AI workflows and automations, enhancing operational efficiency.</span>
* Led a team of <span style="color:red;text-decoration:line-through">4</span> data scientists <span style="color:green">, promoting a collaborative and innovative work environment</span> <span style="color:red;text-decoration:line-through">on enterprise AI initiatives</span>.
<span style="color:red;text-decoration:line-through">* Implemented Retrieval-Augmented Generation systems for internal knowledge management.</span>
* <span style="color:green">Participated in continuous research and benchmarking of AI tools and technologies.</span>

Achievements:

<span style="color:red;text-decoration:line-through">* Reduced model deployment time by 40%.</span>
* <span style="color:green">Improved model prediction accuracy by 11%, contributing to significant cost savings.</span>
<span style="color:red;text-decoration:line-through">* Increased prediction accuracy from 81% to 92%.</span>
* <span style="color:green">Successfully integrated Generative AI into client solutions, enhancing user engagement.</span>
<span style="color:red;text-decoration:line-through">* Generated estimated annual savings of $1.2M through predictive maintenance solutions.</span>

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Responsibilities:

* <span style="color:red;text-decoration:line-through">Built recommendation engines using collaborative filtering techniques.</span>
* <span style="color:red;text-decoration:line-through">Developed fraud detection models using XGBoost and Random Forest algorithms.</span>
* Supported data and AI-related client projects through comprehensive data analysis and insight extraction.
* <span style="color:red;text-decoration:line-through">Performed exploratory data analysis on multi-million record datasets.</span>
* <span style="color:red;text-decoration:line-through">Created executive dashboards for business stakeholders.</span>
* <span style="color:green">Developed prototypes and proof-of-concepts for client solutions.</span>
* <span style="color:green">Engaged in brainstorming sessions on AI adoption and digital transformation.</span>

Achievements:

<span style="color:red;text-decoration:line-through">* Improved fraud detection precision by 25%.</span>
<span style="color:red;text-decoration:line-through">* Reduced reporting time by 70% through automation.</span>
* <span style="color:green">Delivered actionable insights that improved client decision-making processes.</span>
* <span style="color:green">Achieved a 25% improvement in fraud detection precision, enhancing security measures.</span>
<span style="color:red;text-decoration:line-through">* Delivered 15+ production-grade machine learning solutions.</span>

---

## PROJECTS

Enterprise RAG Knowledge Assistant

Technologies:
Python, LangChain, FAISS, OpenAI API<span style="color:red;text-decoration:line-through">, Hugging Face</span>

Description:
Built an enterprise knowledge assistant <span style="color:red;text-decoration:line-through">capable of answering questions from internal company documentation</span> using Retrieval-Augmented Generation <span style="color:green">to optimize internal processes</span>.

Key Results:

* <span style="color:red;text-decoration:line-through">Reduced information retrieval time</span> <span style="color:green">Enhanced information retrieval efficiency</span> by 85%.
<span style="color:red;text-decoration:line-through">* Supported over 5,000 internal documents.</span>

Customer Churn Prediction System

Technologies:
Python, Scikit-learn<span style="color:red;text-decoration:line-through">, XGBoost, Power BI</span>

Description:
Developed a <span style="color:red;text-decoration:line-through">predictive</span> model to identify <span style="color:red;text-decoration:line-through">customers likely to leave subscription services</span> <span style="color:green">at-risk customers, supporting retention strategies</span>.

Results:

* Achieved 93% classification accuracy.
<span style="color:red;text-decoration:line-through">* Improved customer retention strategies.</span>

<span style="color:red;text-decoration:line-through">Medical Image Classification</span>

<span style="color:red;text-decoration:line-through">Technologies:</span>
<span style="color:red;text-decoration:line-through">PyTorch, CNNs, Transfer Learning</span>

<span style="color:red;text-decoration:line-through">Description:</span>
<span style="color:red;text-decoration:line-through">Built a deep learning solution for disease detection from medical images.</span>

<span style="color:red;text-decoration:line-through">Results:</span>

<span style="color:red;text-decoration:line-through">* Achieved 95% validation accuracy.</span>

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate
<span style="color:red;text-decoration:line-through">* TensorFlow Developer Certificate</span>
<span style="color:red;text-decoration:line-through">* Databricks Machine Learning Professional</span>

---

<span style="color:red;text-decoration:line-through">## PUBLICATIONS</span>

<span style="color:red;text-decoration:line-through">Smith, J., "Optimizing Customer Retention Using Machine Learning Models"</span>
<span style="color:red;text-decoration:line-through">International Journal of Data Science, 2023</span>

<span style="color:red;text-decoration:line-through">Smith, J., "Practical Applications of Retrieval-Augmented Generation in Enterprises"</span>
<span style="color:red;text-decoration:line-through">AI Systems Review, 2024</span>

---

<span style="color:red;text-decoration:line-through">## ACHIEVEMENTS</span>

<span style="color:red;text-decoration:line-through">* Winner, Data Science Innovation Challenge 2023</span>
<span style="color:red;text-decoration:line-through">* Speaker at AI & ML Summit 2024</span>
<span style="color:red;text-decoration:line-through">* Top Performer Award, TechNova Analytics (2023)</span>

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

<span style="color:red;text-decoration:line-through">Artificial Intelligence, Open Source Contributions, Data Visualization, Cloud Computing, Research and Innovation</span>
<span style="color:green">AI and Innovation, Digital Transformation, Data Science Trends, Consulting, and Emerging Technologies</span>


--------------------------------
--------------------------------



JOHN ALEXANDER SMITH

Data Scientist | AI Solutions Enthusiast | Innovation Advocate

Email: john.smith@email.com
Phone: +1 (555) 123-4567
Location: Austin, Texas, USA
LinkedIn: linkedin.com/in/johnsmith
GitHub: github.com/johnsmith
Portfolio: www.johnsmithai.com

---

## PROFESSIONAL SUMMARY

Passionate Data Scientist with over 6 years of experience, seeking to leverage my expertise in AI and data science to contribute to innovative projects at Sailpeak. Skilled in Python, data analysis, and machine learning, I excel in transforming complex data into actionable insights. Known for my proactive approach, strong analytical skills, and ability to collaborate effectively in dynamic environments. Committed to continuous improvement and digital transformation in the Banking & Insurance sectors.

---

## CORE SKILLS

Programming Languages:

* Python
* SQL
* R

Machine Learning & AI:

* Supervised Learning
* Unsupervised Learning
* Generative AI Tools
* Feature Engineering
* Model Optimization

Data Analysis & Visualization:

* Pandas
* NumPy
* Matplotlib
* Seaborn
* Power BI

Cloud & MLOps:

* AWS
* Azure
* Docker
* Kubernetes

Soft Skills:

* Strong analytical and problem-solving abilities
* Proactive with a strong sense of ownership
* Excellent communication and collaboration skills

---

## PROFESSIONAL EXPERIENCE

Senior Data Scientist
TechNova Analytics | Austin, Texas
January 2022 - Present

Responsibilities:

* Developed machine learning models for various predictive analytics projects, contributing to improved business outcomes.
* Created AI workflows and automations, enhancing operational efficiency.
* Led a team of data scientists, promoting a collaborative and innovative work environment.
* Participated in continuous research and benchmarking of AI tools and technologies.

Achievements:

* Improved model prediction accuracy by 11%, contributing to significant cost savings.
* Successfully integrated Generative AI into client solutions, enhancing user engagement.

Data Scientist
Insight Data Solutions | Dallas, Texas
June 2019 - December 2021

Responsibilities:

* Supported data and AI-related client projects through comprehensive data analysis and insight extraction.
* Developed prototypes and proof-of-concepts for client solutions.
* Engaged in brainstorming sessions on AI adoption and digital transformation.

Achievements:

* Delivered actionable insights that improved client decision-making processes.
* Achieved a 25% improvement in fraud detection precision, enhancing security measures.

---

## PROJECTS

Enterprise RAG Knowledge Assistant

Technologies:
Python, LangChain, FAISS, OpenAI API

Description:
Built an enterprise knowledge assistant using Retrieval-Augmented Generation to optimize internal processes.

Key Results:

* Enhanced information retrieval efficiency by 85%.

Customer Churn Prediction System

Technologies:
Python, Scikit-learn

Description:
Developed a model to identify at-risk customers, supporting retention strategies.

Results:

* Achieved 93% classification accuracy.

---

## EDUCATION

Master of Science (M.S.) in Data Science
University of Texas at Austin
2017

Bachelor of Science (B.S.) in Computer Science
Texas A&M University
2015

---

## CERTIFICATIONS

* AWS Certified Machine Learning Specialty
* Microsoft Azure Data Scientist Associate

---

## LANGUAGES

* English (Native)
* Spanish (Professional Working Proficiency)

---

## INTERESTS

AI and Innovation, Digital Transformation, Data Science Trends, Consulting, and Emerging Technologies


--------------------------------
--------------------------------



Dear Hiring Manager,

I am writing to express my enthusiasm for the Data Science Intern position at Sailpeak. With over six years of experience in data science and AI, I have honed my skills in transforming complex data into actionable insights, and I am eager to bring my expertise to your innovative consulting firm. Sailpeak's commitment to guiding Banking & Insurance firms through digital transformation aligns perfectly with my passion for AI and continuous improvement in these sectors.

In my current role as a Senior Data Scientist at TechNova Analytics, I have led projects that enhanced operational efficiency through AI workflows and improved model prediction accuracy by 11%, resulting in significant cost savings. My experience in integrating Generative AI into client solutions has also enhanced user engagement, skills that I am excited to leverage at Sailpeak to support your internal and client AI projects. Additionally, my work on enterprise knowledge assistants has optimized information retrieval processes, which I believe will be beneficial in contributing to Sailpeak's AI initiatives.

I am particularly drawn to the opportunity to work in a dynamic and entrepreneurial environment at Sailpeak, where ownership, curiosity, and initiative are highly valued. My proactive approach and strong analytical skills have consistently driven successful outcomes in fast-paced settings, and I am enthusiastic about collaborating with your team to identify AI use cases and improvement opportunities. Moreover, my interest in AI workflows and emerging technologies positions me well to contribute to the design and development of AI workflows and automations at Sailpeak.

I am excited about the prospect of joining Sailpeak and contributing to its mission of digital transformation. I am confident that my experience, coupled with my passion for AI and innovation, will enable me to make a meaningful impact. Thank you for considering my application. I look forward to the opportunity to discuss how I can contribute to your team.

Sincerely,

John Alexander Smith


--------------------------------
--------------------------------

